# 나비야 — 한국어 웨이크워드 학습 (livekit-wakeword 0.2.1)

Colab(GPU) 전용. VoxCPM2로 한국어 "나비야" 합성 → 학습 → **`eval`로 정확도 sanity check**.

설계·배경: `docs/research/d7_wakeword_ko.md`

**이번 목표 = sanity check:** "한국어 나비야가 잡히나"를 livekit 안에서 `eval`로 판정. 우리 데몬
어댑터로의 통합은 그 다음(0.2.1 export는 **livekit-format ONNX**라 우리 openWakeWord 어댑터
직결이 아님 — 통합 시 livekit 런타임용 어댑터 1개 작성, 정확도 확인 후 결정).

**스테이지 분리**로 간다(setup→generate→발음확인→augment→train→export→eval) — 비싼 train 전에
발음을 거르고, 실패 지점을 명확히 하려고.

## 0. 런타임
**런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 선택 후 실행.

In [ ]:
!nvidia-smi

## 1. 시스템 의존성 + 패키지
`voxcpm` = 한국어 TTS. (0.2.1엔 `tflite` extra 없음 — export는 ONNX. `google-adk·starlette` 충돌
경고는 Colab 기존 패키지 문제라 무시.)

In [ ]:
!apt-get -qq install -y espeak-ng libsndfile1 ffmpeg sox portaudio19-dev

In [ ]:
!pip install -q "livekit-wakeword[train,eval,export,voxcpm]"

## 2. config 작성
`tts_backend: voxcpm`(한국어), sanity는 livekit 고정밀 `conv_attention`으로 최선값을 본다.
1차는 작게(스모크) — 통과하면 `n_samples`·`steps`를 키워 본학습.

> ⚠️ config 필드명은 추정 포함. 에러 나면 그 메시지로 한 줄씩 교정.

In [ ]:
import os; os.makedirs('configs', exist_ok=True)

In [ ]:
%%writefile configs/navi_ko.yaml
model_name: navi_ko
data_dir: ./data
target_phrases:
  - "나비야"
tts_backend: voxcpm
n_samples: 2000              # 스모크. 본학습 10000+
model:
  model_type: conv_attention # livekit 고정밀. openWakeWord 호환 필요시 dnn
  model_size: small
steps: 10000                 # 스모크. 본학습 50000

## 3. setup — 외부 의존성 다운로드
VoxCPM HF 스냅샷(한국어) + backgrounds + RIRs. **ACAV100M(~16GB)는 스모크라 `--skip-acav`로 생략**
(negatives는 false-positive에 주로 영향, recall 검증엔 영향 적음). 본학습 땐 빼고 받는다.

> `run`이 negatives 없다고 실패하면 → 이 셀에서 `--skip-acav` 제거하고 재실행.

In [ ]:
!livekit-wakeword setup -c configs/navi_ko.yaml --skip-acav

## 4. generate — "나비야" 합성(양성 + adversarial 음성)
> VoxCPM2 모델 다운로드에서 막히면 HuggingFace 로그인 필요할 수 있음: `!huggingface-cli login`

In [ ]:
!livekit-wakeword generate configs/navi_ko.yaml

## 5. ★ 발음 확인 (train 전 게이트)
"나비야" 합성이 한국어로 멀쩡한지 귀로 확인. 깨지면 여기서 멈추고 잡는다(VoxCPM 설정·언어).

In [ ]:
import glob, IPython.display as ipd
cand = sorted(glob.glob('data/**/*.wav', recursive=True))
pos = [c for c in cand if any(k in c.lower() for k in ('pos', 'navi', 'positive'))] or cand
print(f'wav {len(cand)}개 발견. 미리듣기 {min(3,len(pos))}개:')
for c in pos[:3]:
    print(c); ipd.display(ipd.Audio(c))

## 6. augment → train → export
augment=증강+특징추출, train=분류기 학습(3-phase), export=ONNX. T4에서 스모크는 수십 분.
(한 번에 가려면 위 generate부터 `!livekit-wakeword run configs/navi_ko.yaml` 하나로도 됨.)

In [ ]:
!livekit-wakeword augment configs/navi_ko.yaml

In [ ]:
!livekit-wakeword train configs/navi_ko.yaml

In [ ]:
# 0.2.1 export는 ONNX 전용(--format 없음, --quantize만). run을 쓰면 이미 export됨.
!livekit-wakeword export configs/navi_ko.yaml

## 7. ★ eval — sanity check의 답
recall·FPPH·DET curve. **이 숫자가 "한국어 나비야가 되나"의 판정**이다.

In [ ]:
!livekit-wakeword eval configs/navi_ko.yaml

## 8. ONNX 다운로드 (livekit-format)
통합 결정 전이라도, 받아두면 로컬에서 livekit 런타임으로 마이크 테스트 가능.

In [ ]:
import glob
from google.colab import files
onnx = sorted(glob.glob('**/*.onnx', recursive=True))
onnx = [o for o in onnx if 'navi_ko' in o] or onnx
print('onnx 후보:', onnx)
if onnx:
    files.download(onnx[-1])

## 9. 다음
- **본학습:** `eval`이 양호하면 config를 `n_samples: 10000~50000`·`steps: 50000`로 키우고 setup에서
  `--skip-acav` 제거(negatives 16GB 포함) 후 3~7절 재실행.
- **통합:** 0.2.1 export는 livekit-format ONNX → 우리 데몬엔 livekit 런타임용 어댑터(`WakeWord` 계약
  뒤, 엔진 교체=어댑터1+팩토리 한 줄) 또는 openWakeWord 호환 export가 있는 상위 버전 재검토. **정확도
  확인 후 결정.**